### Fetching datasets via govdata ckan catalog

Ziel ist es den tatsächlichen Datensatz via des GovData CKAN Katalogs zu fetchen

In [30]:
import pandas as pd

df  = pd.read_csv("catalog_distributions.csv")

In [32]:
for x in df.distributions[:20]:
    print(x)
    print(len(x))
    print(type(x))

[{'uri': 'https://ckan.govdata.de/dataset/d7ffe65c-c473-46a9-8a03-98bf3fba57e4/resource/af104411-a1b1-4eb4-a96c-c6632e6fbcca', 'title': 'Dynamisches Angebot der Taxi Dortmund eG', 'access_url': 'https://mobilithek.info//offers/636910977297584128#accessURL', 'download_url': None, 'format': None, 'media_type': None}]
316
<class 'str'>
[{'uri': 'https://ckan.govdata.de/dataset/79969ced-ac94-4e50-9c40-a66b8e1a7423/resource/e4b292f3-1204-41a8-919e-15669814ffb8', 'title': 'dynamische Detektordaten Stadt Rüsselsheim', 'access_url': 'https://mobilithek.info//offers/110000000003182000#accessURL', 'download_url': None, 'format': None, 'media_type': None}]
318
<class 'str'>
[{'uri': 'https://ckan.govdata.de/dataset/e26c3311-5e8d-4d95-befd-eb110d93bc7c/resource/aec610eb-dbb7-4046-8a67-ffd1cfc87e8f', 'title': 'dwista - Tafeln  dynamische Wegweiser mit integrierter Stauinformation', 'access_url': 'http://metaver.de/trefferanzeige%3fdocuuid%3dF1C43CA7-9865-4BE5-8626-43E51EB033B7', 'download_url': 'ht

In [11]:
print(list(df.distributions)[0])
print(type(list(df.distributions)[0]))

[{'uri': 'https://ckan.govdata.de/dataset/d7ffe65c-c473-46a9-8a03-98bf3fba57e4/resource/af104411-a1b1-4eb4-a96c-c6632e6fbcca', 'title': 'Dynamisches Angebot der Taxi Dortmund eG', 'access_url': 'https://mobilithek.info//offers/636910977297584128#accessURL', 'download_url': None, 'format': None, 'media_type': None}]
<class 'list'>


pandas recognized distributions column as string, so we need to convert it back to a list

In [ ]:
import ast
df['distributions'] = df['distributions'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])

In [35]:
df['distributions'][0]

[{'uri': 'https://ckan.govdata.de/dataset/d7ffe65c-c473-46a9-8a03-98bf3fba57e4/resource/af104411-a1b1-4eb4-a96c-c6632e6fbcca',
  'title': 'Dynamisches Angebot der Taxi Dortmund eG',
  'access_url': 'https://mobilithek.info//offers/636910977297584128#accessURL',
  'download_url': None,
  'format': None,
  'media_type': None}]

check media types over all distributions

In [36]:
from collections import Counter

media_type_counter = Counter()


def tabular(x):
    if not isinstance(x, list):
        return

    for item in x:
        if not isinstance(item, dict):
            continue

        if item.get("media_type"):
            media_type_counter[item["media_type"]] += 1


df['distributions'].apply(tabular)

print("Media-Type Distribution:")
for media_type, count in media_type_counter.most_common():
    print(f"{count}: {media_type}")

Media-Type Distribution:
135915: application/x-netcdf
52411: https://www.iana.org/assignments/media-types/text/csv
34167: text/csv
26312: application/pdf
25612: application/zip
14814: https://www.iana.org/assignments/media-types/text/xml
11902: https://www.iana.org/assignments/media-types/text/xlsx
10274: text/plain
7826: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
6490: application/xml
6220: https://www.iana.org/assignments/media-types/application/pdf
4947: https://www.iana.org/assignments/media-types/application/json
4120: https://www.iana.org/assignments/media-types/application/zip
2839: https://www.iana.org/assignments/media-types/application/geo+json
2163: application/json
2140: text/html
2125: application/vnd.ms-excel
1927: text/csv+extended
1649: https://www.iana.org/assignments/media-types/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
958: image/tiff
925: https://www.iana.org/assignments/media-types/application/gml+xml
905: https://www.

In [37]:
tabular_media_types = [
    "https://www.iana.org/assignments/media-types/text/csv",
    "text/csv",
    "https://www.iana.org/assignments/media-types/text/xlsx",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    "application/vnd.ms-excel",
    "text/csv+extended",
    "https://www.iana.org/assignments/media-types/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    "https://www.iana.org/assignments/media-types/application/vnd.ms-excel",
    "https://www.iana.org/assignments/media-types/application/vnd.oasis.opendocument.spreadsheet",
    "application/vnd.oasis.opendocument.spreadsheet",
    "text/tab-separated-values",
    "application/xlsx",
    "https://www.iana.org/assignments/media-types/application/json/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
]

csv_media_types = [
    "https://www.iana.org/assignments/media-types/text/csv",
    "text/csv",
    "text/csv+extended"
]

In [38]:
def has_tabular(distributions):
    for dist in distributions:
        if not isinstance(dist, dict):
            continue
        media_type = dist.get("media_type")
        if media_type in tabular_media_types:
            return True
    return False


df['has_tabular'] = df['distributions'].apply(has_tabular)
df["has_tabular"]

0         False
1         False
2         False
3         False
4         False
          ...  
147254    False
147255    False
147256    False
147257    False
147258    False
Name: has_tabular, Length: 147259, dtype: bool

In [40]:
def has_csv(distributions):
    for dist in distributions:
        if not isinstance(dist, dict):
            continue
        media_type = dist.get("media_type")
        if media_type in csv_media_types:
            return True
    return False

df['has_csv'] = df['distributions'].apply(has_csv)
df["has_csv"]


df.query("has_csv == True")

df['distributions'][0]

[{'uri': 'https://ckan.govdata.de/dataset/d7ffe65c-c473-46a9-8a03-98bf3fba57e4/resource/af104411-a1b1-4eb4-a96c-c6632e6fbcca',
  'title': 'Dynamisches Angebot der Taxi Dortmund eG',
  'access_url': 'https://mobilithek.info//offers/636910977297584128#accessURL',
  'download_url': None,
  'format': None,
  'media_type': None}]

choose 50 random datasets

In [41]:
df_csv_samples = df.query("has_csv == True").sample(50, random_state=42)

df_csv_samples

,title,rdf_about,distributions,has_tabular,has_csv
108398,KFÜ-Messwerte,https://opendata.schleswig-holstein.de/dataset...,[{'uri': 'https://opendata.schleswig-holstein....,True,True
116051,Betriebe im Verarbeitenden Gewerbe nach Wirtsc...,https://region.statistik-nord.de/detail_timeli...,[{'uri': 'https://region.statistik-nord.de/det...,True,True
142955,Messergebnisse zur Radioaktivität in: Sojabohn...,http://suche.transparenz.hamburg.de/dataset/db...,[{'uri': 'http://suche.transparenz.hamburg.de/...,True,True
139606,Messergebnisse zur Radioaktivität in: Schweine...,http://suche.transparenz.hamburg.de/dataset/56...,[{'uri': 'http://suche.transparenz.hamburg.de/...,True,True
118595,Geborenen- (+) bzw. Gestorbenenüberschuss (-) ...,https://region.statistik-nord.de/detail_timeli...,[{'uri': 'https://region.statistik-nord.de/det...,True,True
12930,"München, Landeshauptstadt: Bevölkerung",https://open.bydata.de/api/hub/repo/datasets/1...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True
135982,Messergebnisse zur Radioaktivität in: Reinwass...,http://suche.transparenz.hamburg.de/dataset/72...,[{'uri': 'http://suche.transparenz.hamburg.de/...,True,True
16317,Messdaten Gamma-ODL: Station Kornbach,https://open.bydata.de/api/hub/repo/datasets/o...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True
16265,Messdaten Gamma-ODL: Station Dorfprozelten,https://open.bydata.de/api/hub/repo/datasets/o...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True
13979,Sünching: Baufertigstellungen,https://open.bydata.de/api/hub/repo/datasets/3...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True


In [42]:
import numpy as np

np.random.seed(42)

def select_random_csv(distributions):
    if not isinstance(distributions, list):
        return None

    csv_distributions = [
        item for item in distributions
        if isinstance(item, dict) and item.get("media_type") and "csv" in item["media_type"].lower()
    ]

    if not csv_distributions:
        return None

    return csv_distributions[np.random.randint(len(csv_distributions))]

df_csv_samples["study_distribution"] = df_csv_samples["distributions"].apply(select_random_csv)

In [45]:
df_csv_samples["study_distribution"]

108398    {'uri': 'https://opendata.schleswig-holstein.d...
116051    {'uri': 'https://region.statistik-nord.de/deta...
142955    {'uri': 'http://suche.transparenz.hamburg.de/d...
139606    {'uri': 'http://suche.transparenz.hamburg.de/d...
118595    {'uri': 'https://region.statistik-nord.de/deta...
12930     {'uri': 'https://open.bydata.de/api/hub/repo/d...
135982    {'uri': 'http://suche.transparenz.hamburg.de/d...
16317     {'uri': 'https://open.bydata.de/api/hub/repo/d...
16265     {'uri': 'https://open.bydata.de/api/hub/repo/d...
13979     {'uri': 'https://open.bydata.de/api/hub/repo/d...
15539     {'uri': 'https://open.bydata.de/api/hub/repo/d...
110165    {'uri': 'https://opendata.schleswig-holstein.d...
119718    {'uri': 'https://region.statistik-nord.de/deta...
62290     {'uri': 'https://region.statistik-nord.de/deta...
12586     {'uri': 'https://open.bydata.de/api/hub/repo/d...
104336    {'uri': 'https://opendata.schleswig-holstein.d...
66622     {'uri': 'https://www.landesdat

In [46]:
df_csv_samples.to_csv("../../playground/notebooks/csv_example_datasets.csv", index=True)

In [ ]:
import os
import json
import random
import time
import requests
from pathlib import Path
from typing import Any, Dict, List, Optional

output_dir = Path("../../data")
output_dir.mkdir(exist_ok=True)

# CKAN API base URL (govdata.de)
CKAN_API_BASE = "https://ckan.govdata.de/api/3/action"

# CSV media types to filter for
CSV_MEDIA_TYPES = [
    "https://www.iana.org/assignments/media-types/text/csv",
    "text/csv",
    "text/csv+extended",
]


def create_session():
    """Create a configured requests session."""
    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": "govdata-ckan-client/0.1",
            "Accept": "application/json",
        }
    )
    return session


def get_catalog_count(session: requests.Session) -> int:
    """Get total number of packages in the catalog."""
    url = f"{CKAN_API_BASE}/package_search"
    response = session.get(url, params={"rows": 0})
    response.raise_for_status()
    payload = response.json()
    return payload.get("result", {}).get("count", 0)


def get_random_dataset_names(session: requests.Session, count: int, total: int) -> List[str]:
    """Get random dataset names from the catalog."""
    offsets = random.sample(range(total), min(count, total))
    names = []
    
    for offset in offsets:
        url = f"{CKAN_API_BASE}/package_search"
        response = session.get(url, params={"rows": 1, "start": offset})
        response.raise_for_status()
        payload = response.json()
        results = payload.get("result", {}).get("results", [])
        if results:
            names.append(results[0].get("name"))
        time.sleep(0.1)  # Rate limiting
    
    return names


def fetch_dataset_metadata(session: requests.Session, dataset_name: str) -> Optional[Dict[str, Any]]:
    """Fetch full dataset metadata from CKAN API."""
    url = f"{CKAN_API_BASE}/package_show"
    try:
        response = session.get(url, params={"id": dataset_name}, timeout=30)
        response.raise_for_status()
        payload = response.json()
        if payload.get("success"):
            return payload.get("result")
        return None
    except Exception as e:
        print(f"Error fetching {dataset_name}: {e}")
        return None


def find_valid_csv_distribution(metadata: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Find a valid CSV distribution with a downloadable link."""
    resources = metadata.get("resources", [])
    if not resources:
        return None
    
    # Filter for CSV distributions with valid download URLs
    csv_resources = []
    for res in resources:
        media_type = res.get("mimetype") or res.get("media_type", "")
        download_url = res.get("url", "")
        
        # Check if it's a CSV type
        is_csv = any(csv_type in media_type.lower() for csv_type in CSV_MEDIA_TYPES)
        
        # Check if there's a valid download URL
        has_valid_url = download_url and not download_url.startswith("http://") or download_url.startswith("https://")
        
        if is_csv and has_valid_url and download_url:
            csv_resources.append(res)
    
    # Return a random CSV resource if found
    if csv_resources:
        return random.choice(csv_resources)
    
    return None


def download_file(session: requests.Session, url: str, filepath: Path, label: str) -> bool:
    """Download a file from URL to filepath."""
    try:
        print(f"{label} - Downloading: {url[:60]}...")
        response = session.get(url, timeout=90)
        response.raise_for_status()
        filepath.write_bytes(response.content)
        print(f"{label} - SUCCESS")
        return True
    except Exception as e:
        print(f"{label} - FAILED: {e}")
        return False


def convert_to_xml(metadata: Dict[str, Any]) -> str:
    """Convert metadata dictionary to XML format (simple representation)."""
    import xml.etree.ElementTree as ET
    
    root = ET.Element("metadata")
    
    # Add top-level fields
    for key in ["id", "name", "title", "notes", "license_id", "metadata_created", "metadata_modified"]:
        if metadata.get(key):
            child = ET.SubElement(root, key)
            child.text = str(metadata[key])
    
    # Add organization
    org = metadata.get("organization", {})
    if org:
        org_elem = ET.SubElement(root, "organization")
        for key in ["id", "name", "title"]:
            if org.get(key):
                child = ET.SubElement(org_elem, key)
                child.text = str(org[key])
    
    # Add resources
    for res in metadata.get("resources", []):
        res_elem = ET.SubElement(root, "resource")
        for key in ["id", "name", "url", "format", "mimetype", "size"]:
            if res.get(key):
                child = ET.SubElement(res_elem, key)
                child.text = str(res[key])
    
    # Add tags
    for tag in metadata.get("tags", []):
        tag_elem = ET.SubElement(root, "tag")
        name = tag.get("name", "")
        if name:
            tag_elem.text = str(name)
    
    # Add extras
    for extra in metadata.get("extras", []):
        extra_elem = ET.SubElement(root, "extra")
        key_elem = ET.SubElement(extra_elem, "key")
        key_elem.text = extra.get("key", "")
        val_elem = ET.SubElement(extra_elem, "value")
        val_elem.text = extra.get("value", "")
    
    return ET.tostring(root, encoding="unicode", method="xml")


# Main execution
session = create_session()

# Get total count
print("Getting catalog count...")
total_count = get_catalog_count(session)
print(f"Total packages in catalog: {total_count}")

# Target number of datasets
TARGET_COUNT = 30
ATTEMPT_COUNT = TARGET_COUNT * 3  # Try more in case some fail

print(f"\nFetching {ATTEMPT_COUNT} random dataset names...")
dataset_names = get_random_dataset_names(session, ATTEMPT_COUNT, total_count)

saved_count = 0
attempted = 0

print(f"\nProcessing datasets (target: {TARGET_COUNT})...")

for dataset_name in dataset_names:
    if saved_count >= TARGET_COUNT:
        break
    
    attempted += 1
    print(f"\n--- Attempt {attempted}: {dataset_name} ---")
    
    # Fetch metadata
    metadata = fetch_dataset_metadata(session, dataset_name)
    if not metadata:
        print(f"Skipping {dataset_name}: Could not fetch metadata")
        continue
    
    # Find valid CSV distribution
    csv_resource = find_valid_csv_distribution(metadata)
    if not csv_resource:
        print(f"Skipping {dataset_name}: No valid CSV distribution found")
        continue
    
    download_url = csv_resource.get("url")
    if not download_url:
        print(f"Skipping {dataset_name}: No download URL")
        continue
    
    # Create folder
    folder_name = metadata.get("id", dataset_name)[:50]  # Use ID or truncated name
    row_dir = output_dir / folder_name
    row_dir.mkdir(exist_ok=True)
    
    # Download CSV
    csv_path = row_dir / "dataset.csv"
    if not download_file(session, download_url, csv_path, f"{dataset_name}"):
        continue
    
    # Save metadata as XML
    xml_content = convert_to_xml(metadata)
    xml_path = row_dir / "metadata.xml"
    xml_path.write_text(xml_content, encoding="utf-8")
    print(f"Saved metadata.xml")
    
    saved_count += 1
    print(f"✓ Saved dataset {saved_count}: {dataset_name}")
    
    time.sleep(0.3)  # Rate limiting

session.close()

print(f"\n{'='*50}")
print(f"Done! Saved {saved_count} datasets to {output_dir}")
print(f"Attempted: {attempted}, Success rate: {saved_count/max(attempted,1)*100:.1f}%")

Getting catalog count...
Total packages in catalog: 150624

Fetching 90 random dataset names...

Processing datasets (target: 30)...

--- Attempt 1: kartierungskorridor-ulr ---
Skipping kartierungskorridor-ulr: No valid CSV distribution found

--- Attempt 2: gzr-daten-zur-schwarzarbeit-2015-5-abs-1-und-2-23-abs-1-und-2-aentg-nat-gew ---
Skipping gzr-daten-zur-schwarzarbeit-2015-5-abs-1-und-2-23-abs-1-und-2-aentg-nat-gew: No valid CSV distribution found

--- Attempt 3: zweiter-vorname-mannlich-2025 ---
Skipping zweiter-vorname-mannlich-2025: No valid CSV distribution found

--- Attempt 4: schulzengarten ---
Skipping schulzengarten: No valid CSV distribution found

--- Attempt 5: am-siedlungsweg-kirchweyhe-ursprungsplan ---
Skipping am-siedlungsweg-kirchweyhe-ursprungsplan: No valid CSV distribution found

--- Attempt 6: tagesparameterabhangiges-lastprofil-speicherheizung-ab-2013-berlinf87fb ---
Skipping tagesparameterabhangiges-lastprofil-speicherheizung-ab-2013-berlinf87fb: No valid CS